# Collaborative filtering: user-based and item-based

This notebook implements and evaluates user-based and item-based collaborative filtering using cosine similarity on the sparse user-item matrix.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

from src.data_loader import load_data, build_user_item_matrix, train_test_split_ratings

users, items, ratings = load_data()
train_ratings, test_ratings = train_test_split_ratings(ratings)

## Build sparse user-item matrix

In [ ]:
train_matrix, u2i, i2i, i2u, i2item = build_user_item_matrix(train_ratings)
print(f"Matrix dtype: {train_matrix.dtype}")
print(f"Memory usage: {train_matrix.data.nbytes / 1024:.1f} KB (sparse)")
dense_size = train_matrix.shape[0] * train_matrix.shape[1] * 4
print(f"Dense equivalent would use: {dense_size / (1024*1024):.1f} MB")

## User-user similarity

In [ ]:
# Compute user-user similarity for a sample
sample_size = 100
sample_matrix = train_matrix[:sample_size]
user_sim = cosine_similarity(sample_matrix)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(user_sim, cmap="YlOrRd", ax=ax, vmin=0, vmax=1)
ax.set_title(f"User-user cosine similarity ({sample_size} users)")
ax.set_xlabel("User index")
ax.set_ylabel("User index")
plt.tight_layout()
plt.show()

# Distribution of similarities
upper_tri = user_sim[np.triu_indices_from(user_sim, k=1)]
print(f"Similarity stats -- mean: {upper_tri.mean():.4f}, std: {upper_tri.std():.4f}")
print(f"Min: {upper_tri.min():.4f}, Max: {upper_tri.max():.4f}")

## User-based CF implementation

In [ ]:
def user_based_predict(train_matrix, user_idx, item_idx, n_neighbors=20):
    """Predict a single rating using user-based CF."""
    user_vec = train_matrix[user_idx]
    sims = cosine_similarity(user_vec, train_matrix).flatten()
    sims[user_idx] = 0
    
    # Only consider users who rated this item
    item_ratings = train_matrix[:, item_idx].toarray().flatten()
    rated_mask = item_ratings > 0
    
    if rated_mask.sum() == 0:
        return 0
    
    # Top-k neighbors who rated this item
    valid_sims = sims * rated_mask
    top_k_idx = np.argsort(valid_sims)[::-1][:n_neighbors]
    top_sims = valid_sims[top_k_idx]
    top_ratings = item_ratings[top_k_idx]
    
    if top_sims.sum() == 0:
        return 0
    
    return np.dot(top_sims, top_ratings) / (np.abs(top_sims).sum() + 1e-8)


# Test on a few examples
test_sample = test_ratings.head(20)
preds = []
actuals = []

for _, row in test_sample.iterrows():
    uid, iid, actual = row["user_id"], row["item_id"], row["rating"]
    if uid in u2i and iid in i2i:
        pred = user_based_predict(train_matrix, u2i[uid], i2i[iid])
        preds.append(pred)
        actuals.append(actual)
        print(f"User {uid}, Item {iid}: actual={actual}, predicted={pred:.2f}")

from sklearn.metrics import mean_squared_error, mean_absolute_error
if preds:
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae = mean_absolute_error(actuals, preds)
    print(f"\nSample RMSE: {rmse:.3f}, MAE: {mae:.3f}")

## Item-item similarity

In [ ]:
# Item-item similarity
sample_items = 100
item_sim = cosine_similarity(train_matrix[:, :sample_items].T)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(item_sim, cmap="YlOrRd", ax=ax, vmin=0, vmax=1)
ax.set_title(f"Item-item cosine similarity ({sample_items} items)")
ax.set_xlabel("Item index")
ax.set_ylabel("Item index")
plt.tight_layout()
plt.show()

## Item-based CF implementation

In [ ]:
# Full item similarity (can be large)
full_item_sim = cosine_similarity(train_matrix.T)
print(f"Item similarity matrix shape: {full_item_sim.shape}")

def item_based_predict(train_matrix, item_sim, user_idx, item_idx, n_neighbors=20):
    """Predict a single rating using item-based CF."""
    user_ratings = train_matrix[user_idx].toarray().flatten()
    rated_items = np.where(user_ratings > 0)[0]
    
    if len(rated_items) == 0:
        return 0
    
    sim_scores = item_sim[item_idx, rated_items]
    top_k = np.argsort(sim_scores)[::-1][:n_neighbors]
    
    top_sims = sim_scores[top_k]
    top_ratings = user_ratings[rated_items[top_k]]
    
    if np.abs(top_sims).sum() == 0:
        return 0
    
    return np.dot(top_sims, top_ratings) / (np.abs(top_sims).sum() + 1e-8)


# Test
preds_item = []
actuals_item = []

for _, row in test_sample.iterrows():
    uid, iid, actual = row["user_id"], row["item_id"], row["rating"]
    if uid in u2i and iid in i2i:
        pred = item_based_predict(train_matrix, full_item_sim, u2i[uid], i2i[iid])
        preds_item.append(pred)
        actuals_item.append(actual)

if preds_item:
    rmse = np.sqrt(mean_squared_error(actuals_item, preds_item))
    mae = mean_absolute_error(actuals_item, preds_item)
    print(f"Item-based CF -- RMSE: {rmse:.3f}, MAE: {mae:.3f}")

## Effect of neighborhood size

In [ ]:
# Evaluate user-based CF with different neighborhood sizes
k_values = [5, 10, 20, 30, 50]
eval_sample = test_ratings.sample(200, random_state=42)

results_k = []
for k in k_values:
    preds_k = []
    actuals_k = []
    for _, row in eval_sample.iterrows():
        uid, iid, actual = row["user_id"], row["item_id"], row["rating"]
        if uid in u2i and iid in i2i:
            pred = user_based_predict(train_matrix, u2i[uid], i2i[iid], n_neighbors=k)
            if pred > 0:
                preds_k.append(pred)
                actuals_k.append(actual)
    if preds_k:
        r = np.sqrt(mean_squared_error(actuals_k, preds_k))
        m = mean_absolute_error(actuals_k, preds_k)
        results_k.append({"k": k, "RMSE": r, "MAE": m, "n_predictions": len(preds_k)})
        print(f"k={k:3d} -- RMSE: {r:.3f}, MAE: {m:.3f} ({len(preds_k)} predictions)")

results_df = pd.DataFrame(results_k)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(results_df["k"], results_df["RMSE"], "o-", color="#3B6FD4", label="RMSE", linewidth=2)
ax.plot(results_df["k"], results_df["MAE"], "s-", color="#E8C230", label="MAE", linewidth=2)
ax.set_xlabel("Number of neighbors (k)")
ax.set_ylabel("Error")
ax.set_title("Effect of neighborhood size on prediction accuracy")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Top-N recommendation quality

In [ ]:
from src.model import user_based_cf, item_based_cf, precision_at_k, recall_at_k, ndcg_at_k

# Evaluate precision, recall, NDCG for a sample of users
eval_users = test_ratings["user_id"].unique()[:100]

ub_prec, ub_rec, ub_ndcg = [], [], []
ib_prec, ib_rec, ib_ndcg = [], [], []

for uid in eval_users:
    if uid not in u2i:
        continue
    user_test = test_ratings[(test_ratings["user_id"] == uid) & (test_ratings["rating"] >= 4)]
    relevant = [i2i[iid] for iid in user_test["item_id"] if iid in i2i]
    if not relevant:
        continue
    
    user_idx = u2i[uid]
    
    # User-based
    ub_recs = user_based_cf(train_matrix, user_idx, top_n=10)
    ub_items = [r[0] for r in ub_recs]
    ub_prec.append(precision_at_k(ub_items, relevant, 10))
    ub_rec.append(recall_at_k(ub_items, relevant, 10))
    ub_ndcg.append(ndcg_at_k(ub_items, relevant, 10))
    
    # Item-based
    ib_recs = item_based_cf(train_matrix, user_idx, top_n=10)
    ib_items = [r[0] for r in ib_recs]
    ib_prec.append(precision_at_k(ib_items, relevant, 10))
    ib_rec.append(recall_at_k(ib_items, relevant, 10))
    ib_ndcg.append(ndcg_at_k(ib_items, relevant, 10))

print(f"User-based CF (top-10):")
print(f"  Precision@10: {np.mean(ub_prec):.4f}")
print(f"  Recall@10:    {np.mean(ub_rec):.4f}")
print(f"  NDCG@10:      {np.mean(ub_ndcg):.4f}")
print(f"\nItem-based CF (top-10):")
print(f"  Precision@10: {np.mean(ib_prec):.4f}")
print(f"  Recall@10:    {np.mean(ib_rec):.4f}")
print(f"  NDCG@10:      {np.mean(ib_ndcg):.4f}")

## Comparison: user-based vs item-based

In [ ]:
comparison = pd.DataFrame({
    "User-based CF": [np.mean(ub_prec), np.mean(ub_rec), np.mean(ub_ndcg)],
    "Item-based CF": [np.mean(ib_prec), np.mean(ib_rec), np.mean(ib_ndcg)],
}, index=["Precision@10", "Recall@10", "NDCG@10"])

fig, ax = plt.subplots(figsize=(10, 5))
comparison.plot(kind="bar", ax=ax, color=["#3B6FD4", "#E8C230"], edgecolor="black")
ax.set_title("User-based vs item-based collaborative filtering")
ax.set_ylabel("Score")
ax.set_ylim(0, max(comparison.max().max() * 1.3, 0.1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(comparison.round(4).to_string())

## Similarity distribution analysis

In [ ]:
# Analyze the distribution of similarities
full_user_sim = cosine_similarity(train_matrix[:200])
user_upper = full_user_sim[np.triu_indices_from(full_user_sim, k=1)]

item_upper = full_item_sim[np.triu_indices_from(full_item_sim, k=1)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(user_upper, bins=50, color="#3B6FD4", edgecolor="black", alpha=0.8, density=True)
axes[0].set_title("User-user similarity distribution")
axes[0].set_xlabel("Cosine similarity")
axes[0].set_ylabel("Density")
axes[0].axvline(user_upper.mean(), color="#E8C230", linestyle="--", label=f"Mean: {user_upper.mean():.3f}")
axes[0].legend()

axes[1].hist(item_upper, bins=50, color="#E8C230", edgecolor="black", alpha=0.8, density=True)
axes[1].set_title("Item-item similarity distribution")
axes[1].set_xlabel("Cosine similarity")
axes[1].set_ylabel("Density")
axes[1].axvline(item_upper.mean(), color="#3B6FD4", linestyle="--", label=f"Mean: {item_upper.mean():.3f}")
axes[1].legend()

plt.tight_layout()
plt.show()

## Summary

Key findings from collaborative filtering:

1. **User-user and item-item similarities are generally low** due to high matrix sparsity (95%), which is typical for recommendation datasets
2. **Neighborhood size of 20 provides a reasonable trade-off** between prediction accuracy and computation cost
3. **Item-based CF tends to be more stable** than user-based CF because item profiles change less frequently than user profiles
4. **Both methods struggle with cold-start** -- users or items with very few ratings produce poor similarity estimates
5. **The sparse matrix representation is critical** -- storing the full dense matrix would require over 100x more memory